# RoboEye — DEX 年齡預測模型訓練

MobileNetV2 + DEX (Deep EXpectation) on UTKFace

**使用方式:**
1. Runtime → Change runtime type → **T4 GPU**
2. 上傳 `UTKFace.zip` (執行第一個 cell 時會跳出上傳按鈕)
3. Runtime → **Run all**
4. 最後一個 cell 會自動下載 `best_model.pth`

In [ ]:
# === 1. 確認 GPU 可用 ===
import torch
assert torch.cuda.is_available(), "請先切換到 GPU runtime！(Runtime → Change runtime type → T4 GPU)"
device = torch.device("cuda")
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# === 2. 上傳並解壓 UTKFace ===
import os
from google.colab import files

if not os.path.exists("data/UTKFace"):
    print("請上傳 UTKFace.zip ...")
    uploaded = files.upload()  # 會跳出上傳按鈕
    zip_name = list(uploaded.keys())[0]
    os.makedirs("data", exist_ok=True)
    !unzip -q "{zip_name}" -d data/
    # 如果解壓後多了一層資料夾，自動修正
    if not os.path.exists("data/UTKFace") and os.path.exists(f"data/{zip_name.replace('.zip', '')}"):
        os.rename(f"data/{zip_name.replace('.zip', '')}", "data/UTKFace")

n_files = len(os.listdir("data/UTKFace"))
print(f"✅ UTKFace: {n_files} 張圖片")

In [ ]:
# === 3. Dataset ===
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms


class UTKFaceDataset(Dataset):
    def __init__(self, root, split="train", transform=None, max_age=100):
        self.root = Path(root)
        self.max_age = max_age
        self.transform = transform or self._default_transform(split)

        self.samples = []
        for fname in sorted(os.listdir(self.root)):
            parts = fname.split("_")
            if len(parts) < 3:
                continue
            try:
                age = min(int(parts[0]), self.max_age)
                gender = int(parts[1])
                if gender not in (0, 1):
                    continue
                self.samples.append((fname, age, gender))
            except ValueError:
                continue

        split_idx = int(len(self.samples) * 0.8)
        if split == "train":
            self.samples = self.samples[:split_idx]
        else:
            self.samples = self.samples[split_idx:]

    @staticmethod
    def _default_transform(split):
        if split == "train":
            return transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.RandomHorizontalFlip(),
                transforms.ColorJitter(brightness=0.2, contrast=0.2),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]),
            ])
        else:
            return transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]),
            ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        fname, age, gender = self.samples[idx]
        img = Image.open(self.root / fname).convert("RGB")
        img = self.transform(img)
        return img, age, gender


train_ds = UTKFaceDataset("data/UTKFace", split="train")
val_ds = UTKFaceDataset("data/UTKFace", split="val")
print(f"✅ Train: {len(train_ds)} / Val: {len(val_ds)}")

In [ ]:
# === 4. Model — MobileNetV2 + DEX ===
import torch.nn as nn
import torchvision.models as models


class AgeGenderModel(nn.Module):
    NUM_AGE_CLASSES = 101

    def __init__(self, pretrained=True):
        super().__init__()
        weights = models.MobileNet_V2_Weights.DEFAULT if pretrained else None
        backbone = models.mobilenet_v2(weights=weights)
        self.features = backbone.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        in_features = 1280
        self.age_head = nn.Sequential(
            nn.Dropout(p=0.2),
            nn.Linear(in_features, self.NUM_AGE_CLASSES),
        )
        self.gender_head = nn.Sequential(
            nn.Dropout(p=0.2),
            nn.Linear(in_features, 2),
        )

    def forward(self, x):
        feat = self.features(x)
        feat = self.pool(feat)
        feat = feat.flatten(1)
        return self.age_head(feat), self.gender_head(feat)

    @staticmethod
    def expected_age(age_logits):
        probs = torch.softmax(age_logits, dim=1)
        ages = torch.arange(0, 101, dtype=torch.float32, device=age_logits.device)
        return (probs * ages).sum(dim=1)


model = AgeGenderModel(pretrained=True).to(device)
print(f"✅ Model loaded on {device}")

In [ ]:
# === 5. 訓練 ===
from torch.utils.data import DataLoader
import time

EPOCHS = 20
BATCH_SIZE = 64
LR = 1e-3

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)

age_criterion = nn.CrossEntropyLoss()
gender_criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam([
    {"params": model.features.parameters(), "lr": LR * 0.1},
    {"params": list(model.age_head.parameters()) + list(model.gender_head.parameters()), "lr": LR},
])
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=8, gamma=0.1)

best_mae = float("inf")
history = []
t0 = time.time()

for epoch in range(1, EPOCHS + 1):
    # --- Train ---
    model.train()
    train_loss, train_mae, train_gc, train_n = 0, 0, 0, 0
    for imgs, ages, genders in train_loader:
        imgs, ages, genders = imgs.to(device), ages.to(device), genders.to(device)
        age_logits, gender_logits = model(imgs)
        loss = age_criterion(age_logits, ages) + gender_criterion(gender_logits, genders)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        bs = imgs.size(0)
        train_loss += loss.item() * bs
        train_n += bs
        with torch.no_grad():
            train_mae += (model.expected_age(age_logits) - ages.float()).abs().sum().item()
            train_gc += (gender_logits.argmax(1) == genders).sum().item()

    # --- Val ---
    model.eval()
    val_loss, val_mae_sum, val_gc, val_n = 0, 0, 0, 0
    with torch.no_grad():
        for imgs, ages, genders in val_loader:
            imgs, ages, genders = imgs.to(device), ages.to(device), genders.to(device)
            age_logits, gender_logits = model(imgs)
            loss = age_criterion(age_logits, ages) + gender_criterion(gender_logits, genders)
            bs = imgs.size(0)
            val_loss += loss.item() * bs
            val_n += bs
            val_mae_sum += (model.expected_age(age_logits) - ages.float()).abs().sum().item()
            val_gc += (gender_logits.argmax(1) == genders).sum().item()

    scheduler.step()

    t_mae = train_mae / train_n
    v_mae = val_mae_sum / val_n
    t_gacc = train_gc / train_n
    v_gacc = val_gc / val_n
    elapsed = time.time() - t0

    history.append({"epoch": epoch, "train_mae": t_mae, "val_mae": v_mae,
                    "train_gacc": t_gacc, "val_gacc": v_gacc})

    tag = ""
    if v_mae < best_mae:
        best_mae = v_mae
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "val_mae": best_mae,
            "val_gender_acc": v_gacc,
        }, "best_model.pth")
        tag = " ⭐ best"

    print(f"Epoch {epoch:02d}/{EPOCHS} ({elapsed:.0f}s) | "
          f"Train MAE: {t_mae:.2f}  Gender: {t_gacc:.1%} | "
          f"Val MAE: {v_mae:.2f}  Gender: {v_gacc:.1%}{tag}")

print(f"\n🎉 完成！最佳 Val MAE: {best_mae:.2f}  (耗時 {time.time()-t0:.0f}s)")

In [ ]:
# === 6. 訓練曲線 ===
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

epochs = [h["epoch"] for h in history]
ax1.plot(epochs, [h["train_mae"] for h in history], label="Train")
ax1.plot(epochs, [h["val_mae"] for h in history], label="Val")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Age MAE")
ax1.legend()
ax1.set_title("Age MAE")
ax1.grid(True)

ax2.plot(epochs, [h["train_gacc"] for h in history], label="Train")
ax2.plot(epochs, [h["val_gacc"] for h in history], label="Val")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Gender Accuracy")
ax2.legend()
ax2.set_title("Gender Accuracy")
ax2.grid(True)

plt.tight_layout()
plt.savefig("training_curve.png", dpi=150)
plt.show()
print("✅ 訓練曲線已儲存為 training_curve.png")

In [ ]:
# === 7. 下載模型 ===
from google.colab import files

ckpt = torch.load("best_model.pth", map_location="cpu", weights_only=True)
print(f"最佳模型: Epoch {ckpt['epoch']}, Val MAE: {ckpt['val_mae']:.2f}, Gender Acc: {ckpt['val_gender_acc']:.1%}")
print("\n下載中...")
files.download("best_model.pth")
files.download("training_curve.png")